## Code for generating merged tickets & assets csv

In [6]:
import sys
import os
sys.path.append(os.path.abspath('../utils'))

import pandas as pd
import numpy as np
from data_aggregation_tools import merge_tickets_assets, convert_cols_to_datetime

df_tickets = pd.read_csv("../../data/V_OM_WORK_TASK.csv")
df_assets = pd.read_csv("../../data/V_OM_WORK_TASK_ASSET.csv")

# Generate merged tickets / assets file
df_tickets_assets = merge_tickets_assets(df_tickets=df_tickets, df_assets=df_assets)
df_tickets_assets = convert_cols_to_datetime(df_tickets_assets)

print(f"shape of df_tickets: {df_tickets.shape}")
print(f"shape of df_assets: {df_assets.shape}")
print(f"shape of df_tickets_assets: {df_tickets_assets.shape}")

# print(df_tickets_assets["BASELINE_START_LTZ"].dtype)



/var/folders/y2/tsyh3wpj3mj7zcv4hfd0knl40000gn/T/ipykernel_67696/1661268426.py:9: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_tickets = pd.read_csv("../../data/V_OM_WORK_TASK.csv")
/var/folders/y2/tsyh3wpj3mj7zcv4hfd0knl40000gn/T/ipykernel_67696/1661268426.py:10: DtypeWarning: Columns (0,3,4,5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_assets = pd.read_csv("../../data/V_OM_WORK_TASK_ASSET.csv")


shape of df_tickets: (227205, 47)
shape of df_assets: (316257, 9)
shape of df_tickets_assets: (316257, 55)


In [7]:
print(f"Number of rows with room number attached: {len(df_tickets_assets)}")
df_tickets_assets = df_tickets_assets.dropna(subset=['SPACE'])
print(f"Number of rows with room number attached: {len(df_tickets_assets)}")

Number of rows with room number attached: 316257
Number of rows with room number attached: 125227


## Rules for individual "object"

**All of the following conditions must hold:**
1. Same service class
2. Same request class
3. Same building 

**Future idea:**
1. Same keywords in description (cosine similarity score?)

If the same "object," as defined above, has multiple work tasks in the same 90-day window, then we flag it as a possible persistent issue (repetitive task).

In [8]:
from repetitive_tasks import detect_repetitive

# Group by "Object"

group_cols_tests_dict = {
    'grouping_1': ['SERVICE_CLASS', 'REQUEST_CLASS', 'BUILDING'],
    'grouping_2': ['SERVICE_CLASS', 'REQUEST_CLASS', 'BUILDING', 'RESPONSIBLE_ORGANIZATION_NAME'],
    'grouping_3': ['SERVICE_CLASS', 'REQUEST_CLASS', 'BUILDING', 'RESPONSIBLE_ORGANIZATION_NAME', 'FLOOR'],
    'grouping_4': ['SERVICE_CLASS', 'BUILDING', 'FLOOR', 'SPACE']
}

num_days = 90

# Sort the merged tickets/assets data for faster processing in the filtering step
df_merged_assets_sorted = df_tickets_assets.sort_values(by=['SERVICE_CLASS', 'REQUEST_CLASS', 'BUILDING', 'BASELINE_START_LTZ']).reset_index(drop=True)

# Filter the tickets by "object" (as defined above by tickets with the same group_cols)
df_tickets_filtered_by_object = detect_repetitive(df_tickets_assets, group_cols[0], num_days, verbose=True)

print(f"Found {len(df_tickets_filtered_by_object)} repetitive tasks by object within {num_days} days.")


90 days 00:00:00
Number of columns pre-corrective: 125227
Number of corrective tasks: 69004
Number of unique objects (groups): 29
Found 48021 repetitive tasks by object within 90 days.
Found 48021 repetitive tasks by object within 90 days.


In [9]:
import matplotlib.pyplot as plt

df_corrective = df_merged_assets_sorted[
        df_merged_assets_sorted['TASK_TYPE'] == 'Corrective' 
    ]

group_sizes = df_corrective.groupby(group_cols).size()

size_distribution = group_sizes.value_counts().sort_index()
size_distribution.index.name = 'Group Size'
size_distribution.name = 'Number of Objects'

print(size_distribution.to_string())

Group Size
1      17522
2       5537
3       2362
4       1215
5        778
6        425
7        310
8        215
9        162
10       118
11        79
12        72
13        51
14        41
15        38
16        42
17        17
18        31
19        15
20        14
21        16
22        14
23        10
24        14
25         6
26         1
27        12
28         8
29         8
30         3
31         5
32         6
33         2
34         3
35         5
36         1
37         2
38         4
39         3
40         1
41         1
43         2
44         1
45         2
46         1
48         4
49         3
50         3
54         1
55         2
57         1
58         1
59         1
66         1
70         3
71         1
72         1
73         2
79         1
81         1
90         2
91         1
94         1
95         1
103        1
107        2
111        1
114        1
119        1
129        1
145        1
157        1
160        1
172        1
201        1
229        1
2

# Total Corrective Tasks: 100624 
*(Without grouping by same space)*

When we don't filter out rows that do not have SPACE's attached and sort using the following groupings:

```python
group_cols_tests_dict = {
    'grouping_1': ['SERVICE_CLASS', 'REQUEST_CLASS', 'BUILDING'],
    'grouping_2': ['SERVICE_CLASS', 'REQUEST_CLASS', 'BUILDING', 'RESPONSIBLE_ORGANIZATION_NAME'],
    'grouping_3': ['SERVICE_CLASS', 'REQUEST_CLASS', 'BUILDING', 'RESPONSIBLE_ORGANIZATION_NAME', 'FLOOR'],
    'grouping_4': ['SERVICE_CLASS', 'BUILDING', 'FLOOR', 'SPACE']
}
```

we find:

## 90 days:
- Using grouping_1:
- Using grouping_2:
- Using grouping_3:
- Using grouping_4:

## 45 days:
- Using grouping_1:
- Using grouping_2:
- Using grouping_3:
- Using grouping_4:


Grouping 4 is clearly a better definition for repetitive tasks — with the other groupings, multiple tickets that fix issues in different spaces may trigger a repetition.